# F-003-3a/3c: YOLOv8n Grayscale 転倒検出モデル 精度改善 + 1ch入力対応 Notebook

Issue #101 (精度改善) と Issue #103 (Grayscale対応) を統合した学習ノートブック。
Grayscale (1ch) 入力で精度改善パラメータを適用し、1回の学習で両方をカバーする。

## 目標
| メトリクス | 現在値 (RGB, 100ep) | 目標値 |
|-----------|---------------------|--------|
| mAP@0.5 | 67.8% | 90%以上 |
| Recall | 59.3% | 80%以上 |
| 入力チャネル | 3 (RGB) | 1 (Grayscale) |

## 改善アプローチ (#101)
1. エポック数増加: 100 -> 300
2. 入力サイズ実験: 192x192 / 320x320 の比較
3. データ拡張強化: mixup, copy-paste, scale拡大
4. 学習率スケジュール改善: Cosine LR, lrf=0.001
5. 信頼度閾値別Precision/Recall分析

## Grayscale対応 (#103)
- カスタムモデル YAML で `ch: 1` を指定
- データセット画像をGrayscale変換してから学習
- 色相/彩度のaugmentationを無効化 (hsv_h=0, hsv_s=0)

## 実験計画
- **Experiment A**: 300ep + 192x192 + Grayscale (ベースライン改善)
- **Experiment B**: 300ep + 320x320 + Grayscale (入力サイズ拡大)
- **Experiment C**: 300ep + 320x320 + Grayscale + aug強化 (フル改善)

## 前提条件
- Google Colab (GPU ランタイム: T4 推奨)
- Google Drive に `fall_detection_dataset.zip` をアップロード済み

## ワークフロー
1. GPU確認・環境構築
2. データセット準備 (Grayscale変換)
3. データセット品質分析
4. 実験設定 + カスタムモデルYAML作成
5. モデル学習 (Grayscale + 精度改善パラメータ)
6. 精度評価 (mAP) + RGB版との比較 + KPI判定
7. ONNX エクスポート
8. TFLite FP32/INT8 変換
9. 入力形状・サイズ検証
10. 実験結果サマリ
11. 成果物ダウンロード

---
## Step 1: GPU確認・環境構築

**重要:** メニューの「ランタイム > ランタイムのタイプを変更」で **GPU (T4)** を選択してください。

### 初回実行時の注意
pip installセル実行後、**ランタイムを再起動**してからStep 1を再度実行してください。
Colabにプリインストールされたnumpyとの互換性問題を回避するためです。
（2回目以降はインストール済みなのですぐ完了します）

In [ ]:
# GPU 確認
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('=== GPU が利用可能です ===')
else:
    print('WARNING: GPU が検出されません。ランタイムを GPU に変更してください。')

In [ ]:
# numpy互換性問題を回避してからインストール
!pip install -q numpy==1.26.4
!pip install -q ultralytics onnx onnx2tf onnxsim
print('=== インストール完了 ===')

---
## Step 2: データセット準備 (Grayscale変換)

### 事前準備 (ローカルPCで実行)

```bash
cd mimamori-sense/dataset/merged
zip -r fall_detection_dataset.zip images/ labels/
```

作成した `fall_detection_dataset.zip` を Google Drive のマイドライブ直下にアップロードしてください。

### Grayscale変換について

UltralyticsのデータローダーはデフォルトでRGB画像を読み込みます。
カスタムモデルYAMLで `ch: 1` を指定した場合、モデルのアーキテクチャは
1ch入力になりますが、データローダー側の変換も必要です。

対応方法:
1. **データセット画像を事前にGrayscale変換** (このNotebookで実施)
2. UltralyticsのデータローダーでRGB画像をロードし、`ch: 1` の場合は
   モデルの最初のConv層がRGBチャネルを平均して1chとして処理する

方法1が確実なため、このNotebookでは事前変換を行います。

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

WORK_DIR = '/content/yolo_grayscale_train'
DATASET_DIR = os.path.join(WORK_DIR, 'dataset')
DATASET_GRAY_DIR = os.path.join(WORK_DIR, 'dataset_gray')
DATASET_ZIP = '/content/drive/MyDrive/fall_detection_dataset.zip'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'作業ディレクトリ: {WORK_DIR}')

# データセット展開
if not os.path.isdir(DATASET_DIR):
    if os.path.isfile(DATASET_ZIP):
        print('データセット展開中...')
        !mkdir -p {DATASET_DIR} && unzip -q {DATASET_ZIP} -d {DATASET_DIR}
        print('展開完了')
    else:
        print(f'ERROR: {DATASET_ZIP} が見つかりません')
        print('Google Drive にアップロードしてください')
else:
    print('データセットは展開済みです')

# 検証
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(DATASET_DIR, 'images', split)
    lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    if os.path.isdir(img_dir):
        img_count = len([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
        lbl_count = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')]) if os.path.isdir(lbl_dir) else 0
        print(f'  {split}: images={img_count}, labels={lbl_count}')
    else:
        print(f'  WARNING: {img_dir} が見つかりません')

In [ ]:
# データセット画像をGrayscale変換
# ラベルファイルはそのままコピー（バウンディングボックス座標は変わらない）

from PIL import Image
import shutil

print('=== Grayscale データセット作成 ===')

converted_count = 0
for split in ['train', 'val', 'test']:
    # images ディレクトリ
    src_img_dir = os.path.join(DATASET_DIR, 'images', split)
    dst_img_dir = os.path.join(DATASET_GRAY_DIR, 'images', split)
    os.makedirs(dst_img_dir, exist_ok=True)

    # labels ディレクトリ (そのままコピー)
    src_lbl_dir = os.path.join(DATASET_DIR, 'labels', split)
    dst_lbl_dir = os.path.join(DATASET_GRAY_DIR, 'labels', split)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    if not os.path.isdir(src_img_dir):
        print(f'  WARNING: {src_img_dir} が見つかりません')
        continue

    # 画像をGrayscale変換
    img_files = [f for f in os.listdir(src_img_dir)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    for img_file in img_files:
        src_path = os.path.join(src_img_dir, img_file)
        dst_path = os.path.join(dst_img_dir, img_file)
        try:
            img = Image.open(src_path).convert('L')  # Grayscale変換
            img.save(dst_path)
            converted_count += 1
        except Exception as e:
            print(f'  ERROR: {img_file}: {e}')

    # ラベルをコピー
    if os.path.isdir(src_lbl_dir):
        lbl_files = [f for f in os.listdir(src_lbl_dir) if f.endswith('.txt')]
        for lbl_file in lbl_files:
            shutil.copy2(
                os.path.join(src_lbl_dir, lbl_file),
                os.path.join(dst_lbl_dir, lbl_file)
            )

    img_count = len([f for f in os.listdir(dst_img_dir)
                     if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    lbl_count = len([f for f in os.listdir(dst_lbl_dir)
                     if f.endswith('.txt')])
    print(f'  {split}: images={img_count}, labels={lbl_count}')

print(f'\n合計 {converted_count} 枚の画像をGrayscaleに変換しました')

# 変換結果の確認（サンプル画像表示）
import matplotlib.pyplot as plt
sample_dir = os.path.join(DATASET_GRAY_DIR, 'images', 'train')
if os.path.isdir(sample_dir):
    samples = sorted(os.listdir(sample_dir))[:3]
    if samples:
        fig, axes = plt.subplots(1, len(samples), figsize=(12, 4))
        if len(samples) == 1:
            axes = [axes]
        for ax, s in zip(axes, samples):
            img = Image.open(os.path.join(sample_dir, s))
            ax.imshow(img, cmap='gray')
            ax.set_title(f'{s}\nmode={img.mode}, size={img.size}')
            ax.axis('off')
        plt.suptitle('Grayscale変換サンプル')
        plt.tight_layout()
        plt.show()

In [ ]:
# data.yaml 作成 (Grayscaleデータセット用)
data_yaml = f"""path: {DATASET_GRAY_DIR}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: person
"""

data_yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print(f'data.yaml 作成完了: {data_yaml_path}')
print()
print(data_yaml)

---
## Step 2.5: データセット品質分析

精度改善の前に、Grayscaleデータセットの品質を分析して問題点を把握する。
BBoxサイズ分布、アスペクト比、augmented画像の割合などを確認する。

In [ ]:
# データセット品質分析
import os
import numpy as np
import matplotlib.pyplot as plt

def analyze_dataset(dataset_dir, split='train'):
    """BBoxサイズ分布、アスペクト比、augmented画像の割合を分析"""
    label_dir = os.path.join(dataset_dir, 'labels', split)
    img_dir = os.path.join(dataset_dir, 'images', split)

    if not os.path.isdir(label_dir):
        print(f'ERROR: {label_dir} が見つかりません')
        return

    label_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.txt')])
    print(f'=== {split} データセット品質分析 ===')
    print(f'ラベルファイル数: {len(label_files)}')

    widths, heights, areas, aspect_ratios = [], [], [], []
    empty_labels = 0
    total_boxes = 0

    for lf in label_files:
        path = os.path.join(label_dir, lf)
        with open(path, 'r') as f:
            lines = f.readlines()
        if len(lines) == 0:
            empty_labels += 1
            continue
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                _, cx, cy, w, h = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                widths.append(w)
                heights.append(h)
                areas.append(w * h)
                if h > 0:
                    aspect_ratios.append(w / h)
                total_boxes += 1

    print(f'総BBox数: {total_boxes}')
    print(f'空ラベル: {empty_labels}')
    print(f'画像あたりBBox数: {total_boxes / max(len(label_files) - empty_labels, 1):.2f}')

    # Augmented画像の割合 (ファイル名に 'aug' が含まれるもの)
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))] if os.path.isdir(img_dir) else []
    aug_count = sum(1 for f in img_files if 'aug' in f.lower())
    print(f'総画像数: {len(img_files)}, augmented: {aug_count} ({aug_count/max(len(img_files),1)*100:.1f}%)')

    if total_boxes == 0:
        print('BBoxが見つかりません。分析をスキップします。')
        return

    widths = np.array(widths)
    heights = np.array(heights)
    areas = np.array(areas)
    aspect_ratios = np.array(aspect_ratios)

    print(f'\n--- BBox統計 (正規化座標) ---')
    print(f'幅   : mean={widths.mean():.4f}, std={widths.std():.4f}, min={widths.min():.4f}, max={widths.max():.4f}')
    print(f'高さ : mean={heights.mean():.4f}, std={heights.std():.4f}, min={heights.min():.4f}, max={heights.max():.4f}')
    print(f'面積 : mean={areas.mean():.6f}, std={areas.std():.6f}')
    print(f'AR   : mean={aspect_ratios.mean():.4f}, std={aspect_ratios.std():.4f}')

    # 小さいBBoxの割合 (192x192入力で6x6ピクセル以下 = 正規化で約0.03)
    small_thresh = 0.03
    small_count = np.sum((widths < small_thresh) | (heights < small_thresh))
    print(f'\n小BBox (w or h < {small_thresh}): {small_count} ({small_count/total_boxes*100:.1f}%)')
    print(f'  -> 192x192入力で約{int(small_thresh*192)}ピクセル以下。Recall低下の原因になりうる。')
    print(f'  -> 320x320入力なら約{int(small_thresh*320)}ピクセル。改善が期待できる。')

    # 可視化
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    axes[0, 0].hist(widths, bins=50, alpha=0.7, color='blue')
    axes[0, 0].set_title('BBox Width Distribution')
    axes[0, 0].set_xlabel('Normalized Width')
    axes[0, 0].axvline(x=small_thresh, color='red', linestyle='--', label=f'Small thresh ({small_thresh})')
    axes[0, 0].legend()

    axes[0, 1].hist(heights, bins=50, alpha=0.7, color='green')
    axes[0, 1].set_title('BBox Height Distribution')
    axes[0, 1].set_xlabel('Normalized Height')
    axes[0, 1].axvline(x=small_thresh, color='red', linestyle='--', label=f'Small thresh ({small_thresh})')
    axes[0, 1].legend()

    axes[1, 0].hist(areas, bins=50, alpha=0.7, color='orange')
    axes[1, 0].set_title('BBox Area Distribution')
    axes[1, 0].set_xlabel('Normalized Area')

    axes[1, 1].hist(aspect_ratios, bins=50, alpha=0.7, color='purple')
    axes[1, 1].set_title('Aspect Ratio Distribution (W/H)')
    axes[1, 1].set_xlabel('Aspect Ratio')

    plt.suptitle(f'{split} Dataset BBox Analysis (n={total_boxes})')
    plt.tight_layout()
    plt.show()

# 分析実行
analyze_dataset(DATASET_GRAY_DIR, 'train')
analyze_dataset(DATASET_GRAY_DIR, 'val')

---
## Step 3: 実験設定 + カスタムモデルYAML作成

### 実験バリエーション

| 実験 | エポック | 入力サイズ | データ拡張 | 説明 |
|------|---------|-----------|-----------|------|
| A | 300 | 192x192 | ベースライン | エポック増加の効果を確認 |
| B | 300 | 320x320 | ベースライン | 入力サイズ拡大の効果を確認 |
| C | 300 | 320x320 | 強化 | フル改善 (mixup, copy_paste, scale拡大) |

全実験で Grayscale (ch=1) を使用する。

### 改善ポイント (旧パラメータ -> 改善パラメータ)

| パラメータ | 旧値 | 改善値 | 理由 |
|-----------|------|--------|------|
| epochs | 100 | 300 | 学習の収束を確保 |
| lrf | 0.01 | 0.001 | 学習終盤の微調整精度向上 |
| cos_lr | False | True | 滑らかな学習率減衰 |
| patience | 50 | 100 | 早期終了の猶予拡大 |
| close_mosaic | 10 | 30 | mosaic解除後の微調整期間延長 |
| mixup (実験C) | 0 | 0.15 | 汎化性能向上 |
| copy_paste (実験C) | 0 | 0.1 | 人物出現パターン多様化 |
| scale (実験C) | 0.5 | 0.9 | スケール変動拡大 |
| translate (実験C) | 0.1 | 0.2 | 位置ばらつき拡大 |
| degrees (実験C) | 10 | 15 | 回転ばらつき拡大 |

### 入力サイズの考慮
- 入力サイズはモデルの重みサイズに影響しない (INT8ファイルサイズは同じ)
- ただし推論時のテンソルアリーナサイズは入力サイズに比例して増加
- 320x320で学習しても、デプロイ時に192x192で推論可能 (若干の精度低下あり)

### 学習方式の選択

| 方式 | 説明 | 精度期待 |
|------|------|----------|
| スクラッチ学習 | COCO事前学習重みなしでゼロから学習 | やや低い |
| 転移学習 (推奨) | YOLOv8n事前学習重みの最初のConv層のみ1ch用に調整 | 高い |

Ultralyticsは `ch` が事前学習重みと異なる場合、最初のConv層の重みを
自動的に調整します (3ch重みの平均を取って1chにする)。

In [ ]:
#############################################
# 実験設定 -- ここを変更して実験を切り替える
#############################################

# 実験バリエーション: 'A', 'B', 'C' を切り替え
EXPERIMENT = 'C'

# 入力チャネル (全実験で Grayscale)
INPUT_CHANNELS = 1

# 学習方式: 'transfer' (推奨) or 'scratch'
TRAIN_MODE = 'transfer'

EXPERIMENT_CONFIG = {
    'A': {
        'name': 'exp_a_300ep_192_gray',
        'desc': '300ep + 192x192 + Grayscale (ベースライン改善)',
        'imgsz': 192,
        'epochs': 300,
        'batch': 64,
        'mixup': 0.0,
        'copy_paste': 0.0,
        'scale': 0.5,
        'translate': 0.1,
        'degrees': 10.0,
    },
    'B': {
        'name': 'exp_b_300ep_320_gray',
        'desc': '300ep + 320x320 + Grayscale (入力サイズ拡大)',
        'imgsz': 320,
        'epochs': 300,
        'batch': 32,     # 320x320はメモリを多く使うためバッチ縮小
        'mixup': 0.0,
        'copy_paste': 0.0,
        'scale': 0.5,
        'translate': 0.1,
        'degrees': 10.0,
    },
    'C': {
        'name': 'exp_c_300ep_320_aug_gray',
        'desc': '300ep + 320x320 + Grayscale + aug強化 (フル改善)',
        'imgsz': 320,
        'epochs': 300,
        'batch': 32,
        'mixup': 0.15,
        'copy_paste': 0.1,
        'scale': 0.9,
        'translate': 0.2,
        'degrees': 15.0,
    },
}

config = EXPERIMENT_CONFIG[EXPERIMENT]
IMG_SIZE = config['imgsz']
EPOCHS = config['epochs']
BATCH_SIZE = config['batch']

print(f'=== 実験 {EXPERIMENT}: {config["desc"]} ===')
print(f'  入力: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (Grayscale)')
print(f'  学習方式: {TRAIN_MODE}')
for key, val in config.items():
    print(f'  {key}: {val}')

In [ ]:
# カスタムモデル YAML を動的に生成
model_yaml_content = f"""# YOLOv8n-Grayscale: Fall detection model (Issue #101 + #103)
# Input: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (Grayscale)
# Experiment: {EXPERIMENT} - {config['desc']}

nc: 1  # person only
ch: {INPUT_CHANNELS}  # Grayscale

scales:
  n: [0.33, 0.25, 1024]  # YOLOv8n standard

# YOLOv8 backbone
backbone:
  - [-1, 1, Conv, [64, 3, 2]]       # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]      # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]      # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]      # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]     # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]        # 9

# YOLOv8 head
head:
  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 6], 1, Concat, [1]]       # cat backbone P4
  - [-1, 3, C2f, [512]]             # 12

  - [-1, 1, nn.Upsample, [None, 2, \"nearest\"]]
  - [[-1, 4], 1, Concat, [1]]       # cat backbone P3
  - [-1, 3, C2f, [256]]             # 15 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]      # cat head P4
  - [-1, 3, C2f, [512]]             # 18 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]       # cat head P5
  - [-1, 3, C2f, [1024]]            # 21 (P5/32-large)

  - [[15, 18, 21], 1, Detect, [nc]]  # Detect(P3, P4, P5)
"""

model_yaml_path = os.path.join(WORK_DIR, 'yolov8n-grayscale-fall.yaml')
with open(model_yaml_path, 'w') as f:
    f.write(model_yaml_content)

print(f'モデルYAML作成完了: {model_yaml_path}')
print()
print(model_yaml_content)

In [ ]:
# モデル構造・パラメータ数の確認
from ultralytics import YOLO

temp_model = YOLO(model_yaml_path)
info = temp_model.info(verbose=True)

# パラメータ数からINT8サイズを推定
n_params = sum(p.numel() for p in temp_model.model.parameters())
estimated_int8_kb = n_params / 1024  # INT8: 1パラメータ = 1バイト

print(f'\n=== モデル情報 ===')
print(f'パラメータ数: {n_params:,}')
print(f'推定INT8サイズ: {estimated_int8_kb:.0f} KB ({estimated_int8_kb/1024:.2f} MB)')
print(f'Arena制約 (432KB) との比較: {"PASS" if estimated_int8_kb <= 432 else "FAIL (小型化が必要)"}')
print(f'入力: {IMG_SIZE}x{IMG_SIZE}x{INPUT_CHANNELS} (Grayscale)')

del temp_model

---
## Step 4: モデル学習 (Grayscale + 精度改善パラメータ)

### 精度改善パラメータ (Issue #101)
- エポック数 300 (100 -> 300): 学習の収束を確保
- Cosine学習率スケジュール: 滑らかな学習率減衰
- 最終学習率 0.001 (0.01 -> 0.001): 学習終盤の微調整精度向上
- 早期終了 patience=100: 早期打ち切りを防止
- close_mosaic=30: mosaic解除後の微調整期間延長
- データ拡張強化 (実験Cのみ): mixup, copy_paste, scale/translate/degrees拡大

### Grayscale対応 (Issue #103)
- カスタムYAML で ch=1 指定
- 色相 (hsv_h=0) と彩度 (hsv_s=0) のaugmentationを無効化
- 明度 (hsv_v=0.4) のみ有効 (Grayscaleで重要)

### 転移学習モード (推奨)
- YOLOv8n COCO事前学習重み (`yolov8n.pt`) をロード
- `ch: 1` との不一致は Ultralytics が自動調整
  (最初のConv層の3ch重みを平均して1chに変換)

**注意:** 300エポックの学習は T4 GPU で約3-6時間かかります。
Colab Pro の使用を推奨します。

### 接続切れからの再開

学習結果はGoogle Drive上 (`/content/drive/MyDrive/yolo_training/`) に自動保存されます。
接続が切れた場合:
1. ランタイムを再起動
2. Step 1 から Step 3 まで順に再実行
3. Step 4 のセルを実行すると、`last.pt` を検出して自動的に学習を再開します

In [ ]:
import os
from ultralytics import YOLO

# Google Drive上に学習出力先を設定 (接続切れ対策)
GDRIVE_TRAIN_DIR = '/content/drive/MyDrive/yolo_training'
os.makedirs(GDRIVE_TRAIN_DIR, exist_ok=True)

# 前回の学習が中断された場合、last.pt から再開
last_pt = os.path.join(GDRIVE_TRAIN_DIR, config['name'], 'weights', 'last.pt')

if os.path.exists(last_pt):
    print('=== 前回の学習を再開 ===')
    print(f'再開ポイント: {last_pt}')
    model = YOLO(last_pt)
    results = model.train(resume=True)
else:
    print('=== 新規学習開始 ===')
    if TRAIN_MODE == 'transfer':
        print('転移学習モード: YOLOv8n COCO事前学習重みを1ch用に調整')
        model = YOLO(model_yaml_path).load('yolov8n.pt')
    else:
        print('スクラッチ学習モード')
        model = YOLO(model_yaml_path)

    print(f'\n実験 {EXPERIMENT}: {config["desc"]}')

    results = model.train(
        data=data_yaml_path,
        epochs=config['epochs'],
        imgsz=config['imgsz'],
        batch=config['batch'],
        device=0,
        workers=2,
        project=GDRIVE_TRAIN_DIR,    # Google Driveに保存
        name=config['name'],
        exist_ok=True,
        # --- 学習パラメータ (精度改善版 #101) ---
        optimizer='SGD',
        lr0=0.01,
        lrf=0.001,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=5.0,
        cos_lr=True,
        # --- 早期終了・mosaic制御 (改善) ---
        patience=100,
        close_mosaic=30,
        # --- データ拡張 (Grayscale向け #103 + 精度改善 #101) ---
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.4,
        degrees=config['degrees'],
        translate=config['translate'],
        scale=config['scale'],
        fliplr=0.5,
        mosaic=1.0,
        mixup=config['mixup'],
        copy_paste=config['copy_paste'],
    )

print(f'\n=== 学習完了 (実験 {EXPERIMENT}: {config["desc"]}) ===')

In [ ]:
# 学習曲線の表示
from IPython.display import Image, display
import os

results_png = os.path.join(GDRIVE_TRAIN_DIR, config['name'], 'results.png')
if os.path.exists(results_png):
    display(Image(filename=results_png, width=800))
else:
    print('学習結果の画像が見つかりません')

---
## Step 5: 精度評価 (mAP) + RGB版との比較 + KPI判定

### 評価のポイント
- **mAP@0.5**: 目標 90% 以上
- **Recall**: 目標 80% 以上 (最重要改善指標)
- 旧モデル (RGB, 100ep) との比較
- KPI達成判定
- 入力サイズが異なる場合 (実験B,C) は192x192での追加評価
- 信頼度閾値別のPrecision/Recall分析

In [ ]:
# best.pt で検証データセットを評価
best_pt = os.path.join(GDRIVE_TRAIN_DIR, config['name'], 'weights', 'best.pt')
model = YOLO(best_pt)

# 学習時のimgszで評価
metrics = model.val(
    data=data_yaml_path,
    imgsz=config['imgsz'],
    batch=config['batch'],
    device=0,
    split='val',
)

print(f'\n=== 評価結果 (実験 {EXPERIMENT}: {config["desc"]}) ===')
print(f'入力サイズ  : {config["imgsz"]}x{config["imgsz"]}x{INPUT_CHANNELS} (Grayscale)')
print(f'mAP@0.5     : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

# 旧モデル (RGB, 100ep) との比較
print(f'\n--- 旧モデル (RGB, 100ep, 192x192) との比較 ---')
old_map50, old_recall, old_precision = 67.8, 59.3, 80.4
new_map50 = metrics.box.map50 * 100
new_recall = metrics.box.mr * 100
new_precision = metrics.box.mp * 100

print(f'mAP@0.5   : {new_map50:.1f}% vs {old_map50}% [{new_map50 - old_map50:+.1f}pt]')
print(f'Recall    : {new_recall:.1f}% vs {old_recall}% [{new_recall - old_recall:+.1f}pt]')
print(f'Precision : {new_precision:.1f}% vs {old_precision}% [{new_precision - old_precision:+.1f}pt]')

# KPI 判定
print(f'\n--- KPI 判定 ---')
map_pass = metrics.box.map50 >= 0.9
recall_pass = metrics.box.mr >= 0.8
print(f'mAP@0.5 >= 90%  : {"PASS" if map_pass else "FAIL"} ({new_map50:.1f}%)')
print(f'Recall >= 80%   : {"PASS" if recall_pass else "FAIL"} ({new_recall:.1f}%)')

if map_pass and recall_pass:
    print('\n>>> 全KPI達成')
else:
    print('\n>>> KPI未達。以下を検討:')
    if not recall_pass:
        print('    - Recallが低い場合: 入力サイズを拡大 (実験B,C)')
        print('    - データ拡張を強化 (実験C)')
        print('    - エポック数を更に増加 (400-500)')
    if not map_pass:
        print('    - 全体精度が低い場合: データセットの品質を再確認')

In [ ]:
# 192x192 での追加評価 (デプロイサイズでの精度確認)
# 学習サイズが320x320の場合、推論時は192x192で使用する可能性がある

if config['imgsz'] != 192:
    print('=== 192x192 での評価 (デプロイサイズ) ===')
    print('(学習サイズと異なるため、精度は若干低下する可能性がある)')
    print()

    metrics_192 = model.val(
        data=data_yaml_path,
        imgsz=192,
        batch=64,
        device=0,
        split='val',
    )

    print(f'\n--- 入力サイズ別比較 ---')
    print(f'{"メトリクス":>12s}  {config["imgsz"]}x{config["imgsz"]}   192x192')
    print(f'{"mAP@0.5":>12s}  {metrics.box.map50*100:6.1f}%  {metrics_192.box.map50*100:6.1f}%')
    print(f'{"Recall":>12s}  {metrics.box.mr*100:6.1f}%  {metrics_192.box.mr*100:6.1f}%')
    print(f'{"Precision":>12s}  {metrics.box.mp*100:6.1f}%  {metrics_192.box.mp*100:6.1f}%')

    degradation = metrics.box.map50 * 100 - metrics_192.box.map50 * 100
    print(f'\n192x192での精度劣化: {degradation:.1f}pt')
    if degradation > 10:
        print('WARNING: 精度劣化が大きい。デプロイ時も320x320の使用を検討してください。')
    else:
        print('192x192でも許容範囲内の精度です。')
else:
    print('学習サイズ=192x192のため、追加評価は不要です。')

# テストデータセットでも評価
print()
test_metrics = model.val(
    data=data_yaml_path,
    imgsz=config['imgsz'],
    batch=config['batch'],
    device=0,
    split='test',
)

print(f'\n=== テストデータ評価結果 (実験 {EXPERIMENT}) ===')
print(f'mAP@0.5     : {test_metrics.box.map50:.4f} ({test_metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95: {test_metrics.box.map:.4f} ({test_metrics.box.map*100:.1f}%)')
print(f'Precision    : {test_metrics.box.mp:.4f}')
print(f'Recall       : {test_metrics.box.mr:.4f}')

In [ ]:
# 信頼度閾値別のPrecision/Recall分析
# Recallが低い場合、推論時のconf閾値を下げることで改善できる可能性がある

print('=== 信頼度閾値別 Precision/Recall 分析 ===')
print('推論時のconf閾値を調整してRecallを改善できるか確認する')
print()

for conf_thresh in [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]:
    m = model.val(
        data=data_yaml_path,
        imgsz=config['imgsz'],
        batch=config['batch'],
        device=0,
        split='val',
        conf=conf_thresh,
        verbose=False,
    )
    print(f'conf={conf_thresh:.2f}  mAP50={m.box.map50*100:5.1f}%  P={m.box.mp*100:5.1f}%  R={m.box.mr*100:5.1f}%')

print()
print('注: Ultralyticsのval()ではconfは主にNMS後のフィルタリングに使用される。')
print('mAP計算自体は全信頼度レベルで行われるため、mAP値は大きく変化しない。')
print('デプロイ時にconf閾値を下げることでRecallを改善できる可能性がある。')

---
## Step 6: ONNX エクスポート

デプロイ時の入力サイズ (192x192) でエクスポートする。
学習時に320x320を使用した場合でも、エクスポート時は192x192に設定可能。

In [ ]:
# ONNX エクスポート
# デプロイ時の入力サイズでエクスポート
DEPLOY_IMGSZ = 192  # デプロイ時の入力サイズ (MCU制約)

model = YOLO(best_pt)

onnx_path = model.export(
    format='onnx',
    imgsz=DEPLOY_IMGSZ,
    opset=11,
    simplify=True,
)

print(f'\nONNX エクスポート完了: {onnx_path}')
print(f'デプロイ入力サイズ: {DEPLOY_IMGSZ}x{DEPLOY_IMGSZ}x{INPUT_CHANNELS}')
print(f'サイズ: {os.path.getsize(onnx_path)/1024:.1f} KB')

# ONNX 入力形状の確認
import onnx
onnx_model = onnx.load(onnx_path)
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'ONNX入力: {inp.name}, shape={shape}')
    if shape[-1] == 1 or shape[1] == 1:
        print('  -> 1ch (Grayscale) 入力を確認')
    else:
        print(f'  -> WARNING: {shape[-1]}ch 入力です。ch=1 が期待されます')

---
## Step 7: TFLite FP32/INT8 変換 (Grayscale対応)

In [ ]:
import numpy as np
import glob
import os
import onnx
from PIL import Image
import tensorflow as tf

FP32_PATH = os.path.join(WORK_DIR, 'model_grayscale_fp32.tflite')
INT8_PATH = os.path.join(WORK_DIR, 'model_grayscale_int8.tflite')

# --- Step 7a: ONNX入力形状の確認 ---
print('=== ONNX 入力形状確認 ===')
onnx_model = onnx.load(onnx_path)
for inp in onnx_model.graph.input:
    shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
    print(f'ONNX入力: {inp.name}, shape={shape} (NCHW)')

onnx_shape = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
onnx_ch = onnx_shape[1]  # NCHW format
print(f'入力チャネル数: {onnx_ch}')

if onnx_ch == 1:
    print('-> ONNX は正しく 1ch です。onnx2tf の変換で問題が発生しています。')
elif onnx_ch == 3:
    print('-> ONNX が 3ch のままです。エクスポート時に ch=1 が反映されていません。')

# --- Step 7b: ONNX が3chの場合、1chに修正 ---
ONNX_1CH_PATH = os.path.join(WORK_DIR, 'model_grayscale_1ch.onnx')

if onnx_ch == 3:
    print('\n=== ONNX 入力を 3ch -> 1ch に修正 ===')
    import copy

    model_1ch = copy.deepcopy(onnx_model)

    # 入力テンソルの形状を変更: [1,3,192,192] -> [1,1,192,192]
    inp_tensor = model_1ch.graph.input[0]
    inp_tensor.type.tensor_type.shape.dim[1].dim_value = 1

    # 最初のConv層の重みを修正 (3ch -> 1ch: チャネル平均)
    first_conv_weight = None
    for init in model_1ch.graph.initializer:
        w = np.array(onnx.numpy_helper.to_array(init))
        if w.ndim == 4 and w.shape[1] == 3:  # [out_ch, in_ch=3, kH, kW]
            first_conv_weight = init
            print(f'最初のConv重み: {init.name}, shape={w.shape}')
            # 3ch重みを平均して1chに
            w_1ch = w.mean(axis=1, keepdims=True)  # [out_ch, 1, kH, kW]
            new_tensor = onnx.numpy_helper.from_array(w_1ch, name=init.name)
            init.CopyFrom(new_tensor)
            print(f'  -> 修正後: shape={w_1ch.shape}')
            break

    onnx.save(model_1ch, ONNX_1CH_PATH)
    print(f'1ch ONNX 保存: {ONNX_1CH_PATH}')
    onnx_path_for_convert = ONNX_1CH_PATH
else:
    onnx_path_for_convert = onnx_path

# --- Step 7c: onnx2tf で SavedModel 変換 ---
SAVED_MODEL_DIR = os.path.join(WORK_DIR, 'saved_model')
print(f'\n=== ONNX -> SavedModel (onnx2tf) ===')
os.system(f'onnx2tf -i {onnx_path_for_convert} -o {SAVED_MODEL_DIR} -osd 2>&1 | tail -15')

# SavedModel の入力形状を確認
if os.path.isdir(SAVED_MODEL_DIR):
    loaded = tf.saved_model.load(SAVED_MODEL_DIR)
    sig = loaded.signatures['serving_default']
    for name, spec in sig.structured_input_signature[1].items():
        print(f'SavedModel入力: {name}, shape={spec.shape}')

# --- Step 7d: FP32 TFLite 変換 ---
print('\n=== FP32 TFLite 変換 ===')
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
tflite_fp32 = converter.convert()
with open(FP32_PATH, 'wb') as f:
    f.write(tflite_fp32)

# FP32 入力形状の確認
interp = tf.lite.Interpreter(model_path=FP32_PATH)
interp.allocate_tensors()
inp_detail = interp.get_input_details()[0]
n, h, w, c = inp_detail['shape']
print(f'FP32 TFLite: {os.path.getsize(FP32_PATH)/1024:.1f} KB')
print(f'入力形状: [{n}, {h}, {w}, {c}] (NHWC)')

if c != 1:
    print(f'ERROR: 入力チャネルが {c} です。1ch変換に失敗しています。')
    print('以降の処理をスキップします。')
else:
    # --- Step 7e: INT8 量子化 ---
    print('\n=== INT8 量子化 (Grayscale) ===')
    cal_dir = os.path.join(DATASET_GRAY_DIR, 'images', 'val')
    cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.jpg')))[:200]
    if not cal_images:
        cal_images = sorted(glob.glob(os.path.join(cal_dir, '*.png')))[:200]
    print(f'キャリブレーション画像: {len(cal_images)}枚')

    def representative_dataset():
        for img_path in cal_images:
            img = Image.open(img_path).convert('L').resize((w, h))
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = arr.reshape(1, h, w, 1)
            yield [arr]

    converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    try:
        int8_model = converter.convert()
        with open(INT8_PATH, 'wb') as f:
            f.write(int8_model)
        print(f'INT8 TFLite: {os.path.getsize(INT8_PATH)/1024:.1f} KB')
        print('[PASS] INT8 量子化成功')
    except Exception as e:
        print(f'INT8 量子化エラー: {e}')

---
## Step 8: 入力形状・サイズ検証

### 受け入れ条件の確認

1. モデルの入力が `[1, 192, 192, 1]` (Grayscale) であること
2. INT8量子化後も正常に動作すること
3. 入力バッファサイズが36,864バイト (192*192*1) であること

In [ ]:
import tensorflow as tf
import numpy as np

ARENA_LIMIT_KB = 432

print('=== 入力形状・サイズ検証 ===')
print()

# サイズ比較表
print('--- ファイルサイズ比較 ---')
for label, path in [('FP32', FP32_PATH), ('INT8', INT8_PATH)]:
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        print(f'{label}: {size_kb:.1f} KB ({size_kb/1024:.2f} MB)')

all_checks_passed = True

if os.path.exists(INT8_PATH):
    print(f'\n=== INT8 モデル詳細 ===')
    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    # 入力チェック
    inp_details = interp.get_input_details()
    print(f'\n--- 入力 ---')
    for i, d in enumerate(inp_details):
        shape = d['shape']
        dtype = d['dtype']
        print(f'  [{i}] {d["name"]} shape={shape} dtype={dtype}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

        # 受け入れ条件1: 入力形状チェック
        expected_shape = [1, DEPLOY_IMGSZ, DEPLOY_IMGSZ, INPUT_CHANNELS]
        if list(shape) == expected_shape:
            print(f'      [PASS] 入力形状: {list(shape)} == {expected_shape}')
        else:
            print(f'      [FAIL] 入力形状: {list(shape)} != {expected_shape}')
            all_checks_passed = False

        # 受け入れ条件: INT8 型チェック
        if dtype == np.int8:
            print(f'      [PASS] データ型: INT8')
        else:
            print(f'      [FAIL] データ型: {dtype} (INT8が期待されます)')
            all_checks_passed = False

        # 入力バッファサイズ
        buf_size = 1
        for s in shape:
            buf_size *= s
        expected_buf = DEPLOY_IMGSZ * DEPLOY_IMGSZ * INPUT_CHANNELS  # 36,864
        print(f'      入力バッファサイズ: {buf_size:,} バイト (期待: {expected_buf:,})')
        if buf_size == expected_buf:
            print(f'      [PASS] 入力バッファサイズ一致')
        else:
            print(f'      [FAIL] 入力バッファサイズ不一致')
            all_checks_passed = False

    # 出力チェック
    out_details = interp.get_output_details()
    print(f'\n--- 出力 ---')
    for i, d in enumerate(out_details):
        print(f'  [{i}] {d["name"]} shape={d["shape"]} dtype={d["dtype"]}')
        qp = d.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([]))
        zp = qp.get('zero_points', np.array([]))
        if len(sc) > 0:
            print(f'      scale={sc[0]:.8f}, zero_point={zp[0]}')

    # RGB版との比較
    int8_kb = os.path.getsize(INT8_PATH) / 1024
    rgb_int8_kb = 3149  # RGB版の参考値
    print(f'\n--- RGB版 (3ch) との比較 ---')
    print(f'RGB版 INT8:  {rgb_int8_kb} KB, 入力バッファ: {192*192*3:,} バイト')
    print(f'Gray版 INT8: {int8_kb:.0f} KB, 入力バッファ: {DEPLOY_IMGSZ*DEPLOY_IMGSZ*1:,} バイト')
    print(f'入力バッファ削減: {(1 - DEPLOY_IMGSZ*DEPLOY_IMGSZ*1 / (192*192*3))*100:.0f}%')

    # 推論テスト
    print(f'\n--- 推論テスト (ダミー入力) ---')
    try:
        inp_idx = inp_details[0]['index']
        dummy = np.zeros(inp_details[0]['shape'], dtype=np.int8)
        interp.set_tensor(inp_idx, dummy)
        interp.invoke()
        for i, d in enumerate(out_details):
            out = interp.get_tensor(d['index'])
            print(f'  出力[{i}]: shape={out.shape}, min={out.min()}, max={out.max()}')
        print(f'  [PASS] 推論正常完了')
    except Exception as e:
        print(f'  [FAIL] 推論エラー: {e}')
        all_checks_passed = False

    # 総合判定
    print(f'\n===============================')
    if all_checks_passed:
        print(f'全チェック PASS: Grayscale (1ch) 入力対応完了')
    else:
        print(f'一部チェック FAIL: 上記のエラーを確認してください')
    print(f'===============================')
else:
    print('INT8モデルが見つかりません。Step 7 を先に実行してください。')

In [ ]:
# MCUコード生成用の情報出力
# MainLoop_obj.cc での tensor 設定に使用する値

if os.path.exists(INT8_PATH):
    import tensorflow as tf
    import numpy as np

    interp = tf.lite.Interpreter(model_path=INT8_PATH)
    interp.allocate_tensors()

    print('=== MCUコード生成用情報 ===')
    print()
    print('// ai_application_config.h 設定値')

    inp = interp.get_input_details()[0]
    n, h, w, c = inp['shape']
    print(f'#define INPUT_HEIGHT   {h}')
    print(f'#define INPUT_WIDTH    {w}')
    print(f'#define INPUT_CHANNELS {c}    // Grayscale')
    print(f'#define INPUT_SIZE     ({h} * {w} * {c})  // = {h*w*c}')

    qp = inp.get('quantization_parameters', {})
    sc = qp.get('scales', np.array([0.0]))
    zp = qp.get('zero_points', np.array([0]))
    print(f'// Input quantization: scale={float(sc[0]):.8f}, zero_point={int(zp[0])}')

    print()
    print('// MainLoop_obj.cc 出力テンソル設定')
    for i, o in enumerate(interp.get_output_details()):
        qp = o.get('quantization_parameters', {})
        sc = qp.get('scales', np.array([0.0]))
        zp = qp.get('zero_points', np.array([0]))
        print(f'// Output[{i}]: shape={list(o["shape"])}, scale={float(sc[0]):.8f}, zero_point={int(zp[0])}')

---
## Step 9: 実験結果サマリ

全実験の結果を比較し、最適な構成を選択する。
各実験後にこのセルを実行して結果を記録する。

In [ ]:
# 実験結果サマリ
# 各実験後にこのセルを実行して結果を記録する

print('=== 実験結果サマリ ===')
print()

# 結果テンプレート (実験ごとに値を更新)
results_table = {
    'Baseline (RGB, 100ep, 192)': {'mAP50': 67.8, 'Recall': 59.3, 'Precision': 80.4, 'INT8_KB': 3149},
    # 実験後に以下を更新
    # 'Exp A (300ep, 192, Gray)': {'mAP50': None, 'Recall': None, 'Precision': None, 'INT8_KB': None},
    # 'Exp B (300ep, 320, Gray)': {'mAP50': None, 'Recall': None, 'Precision': None, 'INT8_KB': None},
    # 'Exp C (300ep, 320, aug, Gray)': {'mAP50': None, 'Recall': None, 'Precision': None, 'INT8_KB': None},
}

# 現在の実験結果を追加
if 'metrics' in dir():
    exp_label = f'Exp {EXPERIMENT} ({config["epochs"]}ep, {config["imgsz"]}, Gray)'
    int8_kb = os.path.getsize(INT8_PATH) / 1024 if os.path.exists(INT8_PATH) else None
    results_table[exp_label] = {
        'mAP50': round(metrics.box.map50 * 100, 1),
        'Recall': round(metrics.box.mr * 100, 1),
        'Precision': round(metrics.box.mp * 100, 1),
        'INT8_KB': int(int8_kb) if int8_kb else None,
    }

print(f'{"実験":>35s}  mAP50  Recall  Prec   INT8(KB)')
print('-' * 75)
for name, r in results_table.items():
    m = f'{r["mAP50"]:5.1f}%' if r['mAP50'] else '  N/A '
    rc = f'{r["Recall"]:5.1f}%' if r['Recall'] else '  N/A '
    p = f'{r["Precision"]:5.1f}%' if r['Precision'] else '  N/A '
    s = f'{r["INT8_KB"]:,}' if r['INT8_KB'] else 'N/A'
    print(f'{name:>35s}  {m}  {rc}  {p}  {s}')

print()
print('目標: mAP50 >= 90%, Recall >= 80%')
print()
print('注意:')
print('- 全実験で Grayscale (ch=1) を使用')
print('- 入力サイズはモデルの重みサイズに影響しない (INT8 KB は同じ)')
print('- 推論時のテンソルアリーナサイズは入力サイズに比例して増加')
print('- 推奨実験順序: A -> B -> C (段階的に改善効果を確認)')

---
## Step 10: 成果物ダウンロード

学習済みモデルを Google Drive に保存する。

In [ ]:
# Google Drive に成果物をコピー
# 注: 学習出力 (best.pt, last.pt, results.*) は既に GDRIVE_TRAIN_DIR に保存済み
#      ここでは変換成果物 (ONNX, TFLite) と設定ファイルを追加コピーする
import shutil

OUTPUT_DIR = f'/content/drive/MyDrive/fall_detection_model_gray_{EXPERIMENT}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 学習出力 (既にGoogle Drive上にあるファイル) -> 別ディレクトリにもコピー
gdrive_files = {
    best_pt: f'best_gray_{EXPERIMENT}.pt',
    os.path.join(GDRIVE_TRAIN_DIR, config['name'], 'weights', 'last.pt'): f'last_gray_{EXPERIMENT}.pt',
    os.path.join(GDRIVE_TRAIN_DIR, config['name'], 'results.png'): 'results.png',
    os.path.join(GDRIVE_TRAIN_DIR, config['name'], 'results.csv'): 'results.csv',
}

# ローカル変換成果物
local_files = {
    onnx_path: f'model_gray_{EXPERIMENT}.onnx',
    FP32_PATH: f'model_gray_{EXPERIMENT}_fp32.tflite',
    INT8_PATH: f'model_gray_{EXPERIMENT}_int8.tflite',
    model_yaml_path: 'yolov8n-grayscale-fall.yaml',
}

files_to_copy = {**gdrive_files, **local_files}

for src, dst_name in files_to_copy.items():
    if os.path.exists(src):
        dst = os.path.join(OUTPUT_DIR, dst_name)
        shutil.copy2(src, dst)
        size_kb = os.path.getsize(src) / 1024
        print(f'  {dst_name}: {size_kb:.1f} KB')

print(f'\n=== Google Drive に保存完了 ===')
print(f'成果物: {OUTPUT_DIR}')
print(f'学習ログ: {GDRIVE_TRAIN_DIR}/{config["name"]}/')

---
## まとめ

### 本ノートブックの位置づけ

Issue #101 (精度改善) と Issue #103 (Grayscale対応) を統合した学習ノートブック。
Grayscale (1ch) 入力で精度改善パラメータを適用し、1回の学習で両方をカバーする。

### 精度改善の変更点 (#101, 旧ノートブックからの差分)

| 変更項目 | 旧値 | 新値 | 改善効果の期待 |
|---------|------|------|---------------|
| エポック数 | 100 | 300 | 学習の収束を確保 |
| 最終学習率 (lrf) | 0.01 | 0.001 | 学習終盤の微調整精度向上 |
| Cosine学習率 | 無効 | 有効 | 滑らかな学習率減衰 |
| ウォームアップ | 3ep | 5ep | 学習初期の安定性 |
| 早期終了patience | 50 | 100 | 学習の打ち切り防止 |
| close_mosaic | 10 | 30 | mosaic解除後の微調整期間延長 |
| 入力サイズ (実験B,C) | 192 | 320 | 小人物の検出改善 |
| mixup (実験C) | 0 | 0.15 | 汎化性能向上 |
| copy_paste (実験C) | 0 | 0.1 | 人物出現パターン多様化 |
| scale (実験C) | 0.5 | 0.9 | スケール変動拡大 |
| translate (実験C) | 0.1 | 0.2 | 位置ばらつき拡大 |
| degrees (実験C) | 10 | 15 | 回転ばらつき拡大 |

### Grayscale対応の変更点 (#103)

| 項目 | RGB版 | Grayscale版 |
|------|-------|-------------|
| 入力形状 | [1, 192, 192, 3] | [1, 192, 192, 1] |
| 入力バッファ | 110,592バイト | 36,864バイト |
| バッファ削減 | - | 66.7% 削減 |
| hsv_h, hsv_s | 有効 | 無効 (0.0) |
| MCUカメラとの整合性 | チャネル複製が必要 | そのまま使用可能 |

### 生成される成果物

| ファイル | 説明 |
|---|---|
| `best_gray_{EXP}.pt` | 学習済み PyTorch モデル (Grayscale, 最良 mAP) |
| `model_gray_{EXP}.onnx` | ONNX 形式モデル (1ch入力) |
| `model_gray_{EXP}_fp32.tflite` | TFLite FP32 モデル (1ch入力) |
| `model_gray_{EXP}_int8.tflite` | TFLite INT8 量子化モデル (1ch入力) |
| `yolov8n-grayscale-fall.yaml` | カスタムモデル設定 |
| `results.png` | 学習曲線チャート |
| `results.csv` | 学習ログ (CSV) |

### 推奨実験順序

1. **まず実験A** (300ep, 192, Gray) を実行し、エポック数増加だけでどこまで改善するか確認
2. 目標未達の場合 **実験B** (300ep, 320, Gray) で入力サイズ拡大の効果を確認
3. さらに改善が必要なら **実験C** (300ep, 320, aug, Gray) を実行

### 次のステップ

1. 精度目標 (mAP >= 90%, Recall >= 80%) が達成できたら、F-003-3b の小型化検討に進む
2. F-003-3b (モデル小型化) と組み合わせて、Grayscale + pico/nano-slim で最終モデルを決定
3. RUHMI/MERA SDK で Ethos-U55 向けに変換
4. 実機 (EK-RA8P1) での動作確認

### 備考

- F-003-3b の pico/nano-slim ノートブック (`train_yolov8_pico_colab.ipynb`) は
  既に `ch: 1` (Grayscale) 対応済みです
- 本ノートブックは YOLOv8n サイズ (width=0.25) での Grayscale 学習です
- YOLOv8n サイズのINT8モデルは約3MB程度になるため、Arena制約 (432KB) を
  満たすにはF-003-3bの小型化が必須です
- `train_yolov8_improved_colab.ipynb` (RGB版精度改善) は本ノートブックに統合済みのため削除